<a href="https://colab.research.google.com/github/MiyoBran/alura-etl-telecom-x-parte-II/blob/main/EST_TelecomX_LATAM_Machine_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Challenge Telecom X Parte II

In [1]:
import pandas as pd
import seaborn as sns

# ==============================================================================
# IMPORTACIÓN DEL DATASET
# ==============================================================================

ruta_original = 'https://github.com/MiyoBran/alura-etl-telecom-x-parte-II/blob/main/datos_tratados.csv'
ruta_original = ruta_original.replace('/github.com' , '/raw.githubusercontent.com')
ruta_original = ruta_original.replace('/blob' , '')
df = pd.read_csv(ruta_original)

# Nuestro Dataset "Original"
display(df.head())


# El Dataset Nuevo , cargado desde url en este punto si se decide obviar todo lo
# anterior, tratado con nuestras modificaciones
ruta_dataset = 'https://github.com/MiyoBran/alura-etl-telecom-x-parte-II/blob/main/dataset_telecom_ML.csv'
ruta_dataset = ruta_dataset.replace('/blob' , '').replace('/github.com' , '/raw.githubusercontent.com')
df_ml = pd.read_csv(ruta_dataset)
df_ml.head()

,ID_Cliente,Evasion,Genero,Adulto_Mayor,Pareja,Dependientes,Meses_Contrato,Servicio_Telefonico,Multiples_Lineas,Servicio_Internet,...,Streaming_TV,Streaming_Peliculas,Contrato,Facturacion_Electronica,Metodo_Pago,Cargo_Mensual,Cargo_Total,Cuentas_Diarias,Puntaje_Inseguro,Total_Servicios
0,0002-ORFBO,0,1,0,1,1,9,1,0,DSL,...,1,0,One year,1,Mailed check,65.6,593.30,2.19,2.0,4.0
1,0003-MKNFE,0,0,0,0,0,9,1,1,DSL,...,0,1,Month-to-month,0,Mailed check,59.9,542.40,2.00,4.0,2.0
2,0004-TLHLJ,1,0,0,0,0,4,1,0,Fiber optic,...,0,0,Month-to-month,1,Electronic check,73.9,280.85,2.46,3.0,2.0
3,0011-IGKFF,1,0,1,1,0,13,1,0,Fiber optic,...,1,1,Month-to-month,1,Electronic check,98.0,1237.85,3.27,2.0,5.0
4,0013-EXCHZ,1,1,1,1,0,3,1,0,Fiber optic,...,1,0,Month-to-month,1,Mailed check,83.9,267.40,2.80,3.0,3.0


,Evasion,Adulto_Mayor,Pareja,Dependientes,Meses_Contrato,Multiples_Lineas,Servicio_Internet,Seguridad_Online,Respaldo_Online,Proteccion_Dispositivo,Soporte_Tecnico,Streaming_TV,Streaming_Peliculas,Contrato,Facturacion_Electronica,Metodo_Pago,Cargo_Mensual,Puntaje_Inseguro,Total_Servicios
0,0,0,1,1,9,0,DSL,0,1,0,1,1,0,One year,1,Mailed check,65.6,2.0,4.0
1,0,0,0,0,9,1,DSL,0,0,0,0,0,1,Month-to-month,0,Mailed check,59.9,4.0,2.0
2,1,0,0,0,4,0,Fiber optic,0,0,1,0,0,0,Month-to-month,1,Electronic check,73.9,3.0,2.0
3,1,1,1,0,13,0,Fiber optic,0,1,1,0,1,1,Month-to-month,1,Electronic check,98.0,2.0,5.0
4,1,1,1,0,3,0,Fiber optic,0,0,0,1,1,0,Month-to-month,1,Mailed check,83.9,3.0,3.0


## Preparando el dataset

In [ ]:
# Asumimos importado pandas as pd y seaborn as sns
# También requerimos importar matplotlib para un control fino de la arquitectura visual
import matplotlib.pyplot as plt

def auditar_distribucion_y_relaciones(df: pd.DataFrame, col_target: str, cols_predictores: list[str]) -> None:
    """
    Ejecuta una auditoría visual de la variable dependiente (Y) y sus predictores (X).
    Evalúa estadísticamente la asimetría poblacional y la varianza (Homocedasticidad vs Heterocedasticidad).
    """
    # 1. Configuración global de estilo (Se define una sola vez para mantener coherencia visual)
    sns.set_palette("Accent")
    sns.set_style("darkgrid")

    # ---------------------------------------------------------------------
    # DIAGNÓSTICO 1: Distribución del Target (Auditoría de Asimetría)
    # ---------------------------------------------------------------------
    # Utilizamos subplots para consolidar el Boxplot y el Histograma en una sola figura limpia
    fig, axes = plt.subplots(1, 2, figsize=(20, 6))
    fig.suptitle('Auditoría del Target (Y): Análisis de Asimetría en USD', fontsize=20, y=1.05)

    # Boxplot: Permite visualizar los outliers (valores atípicos) que "estiran" la media poblacional
    sns.boxplot(data=df, x=col_target, ax=axes[0], width=0.3)
    axes[0].set_title('Boxplot: Concentración Intercuartílica')
    axes[0].set_xlabel('Valor (USD)')

    # Histograma + KDE: Visualiza la Asimetría Positiva (Cola larga hacia la derecha)
    sns.histplot(data=df, x=col_target, kde=True, ax=axes[1])
    axes[1].set_title('Histograma: Distribución de Frecuencias')
    axes[1].set_xlabel('Valor (USD)')

    plt.show()

    # ---------------------------------------------------------------------
    # DIAGNÓSTICO 2: Dispersión y Varianza (Auditoría de Heterocedasticidad)
    # ---------------------------------------------------------------------
    # Pairplot con línea de regresión (kind='reg').
    # Buscamos observar si el error es constante (Homocedasticidad) o si forma un cono (Heterocedasticidad).
    grafico_relaciones = sns.pairplot(
        data=df,
        y_vars=col_target,
        x_vars=cols_predictores,
        kind='reg',
        height=5
    )
    grafico_relaciones.fig.suptitle('Relación Lineal: Predictores (X) vs Target (Y)', fontsize=20, y=1.05)
    plt.show()

# Ejecución de la función con variables claramente definidas
# Supongamos que tu DataFrame ya fue cargado previamente como 'df_propiedades'
variables_predictoras = ['Area', 'Dist_Playa', 'Dist_Farmacia']
auditar_distribucion_y_relaciones(df_propiedades, col_target='Valor', cols_predictores=variables_predictoras)

NameError: name 'df_propiedades' is not defined